# معمل بناء أول تدفق عمل بسيط في LangGraph
### Interactive Hands-on Lab: Pure Python Workflow & Branching (Lecture 73)

أهلاً بك يا بطل!
في هذا المعمل، سنتعلم أساسيات **LangGraph** بطريقة نقية ومبسطة جداً: **بدون أي نماذج ذكاء اصطناعي (No LLMs)** في البداية!
الهدف هنا هو أن تفهم وتجرب بنفسك كيف تتحرك البيانات داخل الجراف وكيف تتخذ القرارات الشرطية.

المشروع البسيط الذي سنبنيه:
1. نبدأ من نقطة البداية (START).
2. نمر على عقدة التخطيط: `start_play` (أنا أخطط للعب).
3. نقرر شرطياً: هل سنلعب `cricket` (كريكت) أم `badminton` (ريشة طائرة)؟
4. نصل إلى نقطة النهاية (END).

جاهز؟ تعال نبدأ خطوة بخطوة!


In [1]:
import random
from typing import Dict, Any, Literal
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

print("تم استيراد كافة أدوات الجراف بنجاح!")


تم استيراد كافة أدوات الجراف بنجاح!


### 1. تعريف الحالة المشتركة (State Schema)
الحالة هي الصندوق أو الشنطة التي تمر على كل عقدة، وكل عقدة تضع فيها معلومة جديدة.
هنا سنعرف كلاس حالة بسيط جداً باستخدام `TypedDict` يحتوي على متغير نصي واحد اسمه `graph_info`.


In [2]:
class SimplePlayState(TypedDict):
    graph_info: str

print("تم تعريف كلاس الحالة SimplePlayState بنجاح!")


تم تعريف كلاس الحالة SimplePlayState بنجاح!


### 2. بناء العقد البرمجية (Nodes)
كل عقدة هي مجرد دالة بايثون عادية جداً:
- تأخذ كائن الحالة الحالي `state`.
- تقوم بعمليتها وتطبع رسالة توضيحية.
- تعيد قاموساً بالتحديث الجديد (Delta).


In [3]:
def start_play(state: SimplePlayState) -> Dict[str, Any]:
    print("[عقدة البداية]: تم استدعاء start_play")
    old_info = state.get("graph_info", "")
    new_info = f"{old_info} أنا أخطط للعب".strip()
    return {"graph_info": new_info}

def cricket_game(state: SimplePlayState) -> Dict[str, Any]:
    print("[عقدة الكريكت]: تم اختيار لعبة الكريكت!")
    old_info = state.get("graph_info", "")
    return {"graph_info": f"{old_info} كريكت"}

def badminton_game(state: SimplePlayState) -> Dict[str, Any]:
    print("[عقدة الريشة]: تم اختيار لعبة الريشة الطائرة!")
    old_info = state.get("graph_info", "")
    return {"graph_info": f"{old_info} ريشة طائرة"}

print("تم تجهيز العقد الثلاث بنجاح!")


تم تجهيز العقد الثلاث بنجاح!


### 3. دالة اتخاذ القرار الشرطي (Conditional Router)
هذه الدالة تفحص الحالة أو تختار مساراً عشوائياً، وتعيد اسم العقدة التالية بدقة (`Literal`).


In [4]:
def decide_game(state: SimplePlayState) -> Literal["cricket_game", "badminton_game"]:
    rand_val = random.random()
    if rand_val > 0.5:
        chosen = "cricket_game"
    else:
        chosen = "badminton_game"
    print(f"[دالة التوجيه الشرطي]: القيمة العشوائية كانت {rand_val:.2f} -> التوجيه إلى: {chosen}")
    return chosen

print("تم تجهيز دالة التوجيه الشرطي بنجاح!")


تم تجهيز دالة التوجيه الشرطي بنجاح!


### 4. تجميع الجراف وتوصيل الأسهم (Edges)
الآن سنربط العقد مع نقطة البداية والنهاية والتفرع الشرطي.


In [5]:
workflow = StateGraph(SimplePlayState)

# 1. إضافة العقد
workflow.add_node("start_play", start_play)
workflow.add_node("cricket_game", cricket_game)
workflow.add_node("badminton_game", badminton_game)

# 2. حافة مباشرة من START إلى start_play
workflow.add_edge(START, "start_play")

# 3. حافة شرطية من start_play إلى اللعبة المختارة
workflow.add_conditional_edges(
    "start_play",
    decide_game,
    {
        "cricket_game": "cricket_game",
        "badminton_game": "badminton_game"
    }
)

# 4. توصيل نهايات اللعبتين بنقطة النهاية END
workflow.add_edge("cricket_game", END)
workflow.add_edge("badminton_game", END)

# 5. التجميع (Compile)
game_app = workflow.compile()

print("تم بناء وتجميع الجراف بنجاح، وهو جاهز للتشغيل!")


تم بناء وتجميع الجراف بنجاح، وهو جاهز للتشغيل!


### 5. تشغيل وتجربة الجراف عملياً
تعال نجرب تشغيل الجراف ونرى كيف تتدفق الكلمات عبر الحالة.


In [6]:
start_data: SimplePlayState = {"graph_info": "بداية التدفق ->"}

print("تشغيل الجراف خطوة بخطوة:")
print("-" * 50)

for step in game_app.stream(start_data):
    for node_name, result in step.items():
        print(f"العقدة المنفذة: [{node_name}] -> الحالة الحالية: {result}")

print("-" * 50)
final_state = game_app.invoke(start_data)
print("النتيجة النهائية المسجلة في الحالة:")
print(final_state["graph_info"])


تشغيل الجراف خطوة بخطوة:
--------------------------------------------------
[عقدة البداية]: تم استدعاء start_play
[دالة التوجيه الشرطي]: القيمة العشوائية كانت 0.59 -> التوجيه إلى: cricket_game
العقدة المنفذة: [start_play] -> الحالة الحالية: {'graph_info': 'بداية التدفق -> أنا أخطط للعب'}
[عقدة الكريكت]: تم اختيار لعبة الكريكت!
العقدة المنفذة: [cricket_game] -> الحالة الحالية: {'graph_info': 'بداية التدفق -> أنا أخطط للعب كريكت'}
--------------------------------------------------
[عقدة البداية]: تم استدعاء start_play
[دالة التوجيه الشرطي]: القيمة العشوائية كانت 0.37 -> التوجيه إلى: badminton_game
[عقدة الريشة]: تم اختيار لعبة الريشة الطائرة!
النتيجة النهائية المسجلة في الحالة:
بداية التدفق -> أنا أخطط للعب ريشة طائرة


### 6. جرب بنفسك! (Hands-on Experiments)
1. **تحدي 1**: أضف عقدة رابعة باسم `football_game` تضيف "كرة قدم".
2. **تحدي 2**: عدل دالة `decide_game` لتختار بين 3 ألعاب (كريكت، ريشة، كرة قدم) بدلاً من اثنتين!
3. **تحدي 3**: جرب إضافة متغير جديد في الـ `SimplePlayState` باسم `score: int` واجعل كل عقدة تضيف 10 نقاط للنتيجة وشاهد المجموع في النهاية!
